# Model Implementation

In this notebook, we instantiate the three models:
1. **RNN-AVG**: Shared BLSTM with mean pooling.
2. **RNN-ATT**: Attention-BLSTM (classical attention without positional info).
3. **RNN-POA**: Positional-Attention model from Chen et al. (2017).

In [ ]:
import sys
sys.path.append('..')

import torch
import numpy as np
from utils.model_utils import RNNAVG, RNNATT, RNNPOA

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### Create Dummy Data & Embeddings for Testing

In [ ]:
VOCAB_SIZE = 1000
EMBED_DIM = 100
HIDDEN_DIM = 50

dummy_embed_matrix = np.random.uniform(-0.25, 0.25, (VOCAB_SIZE, EMBED_DIM)).astype(np.float32)
dummy_embed_matrix[0] = np.zeros(EMBED_DIM)

### 1. RNN-AVG
A simple baseline that encodes both sequences with a shared BLSTM, mean-pools the outputs, and computes Manhattan similarity.

In [ ]:
model_avg = RNNAVG(dummy_embed_matrix, hidden_dim=HIDDEN_DIM).to(device)
print(model_avg)

### 2. RNN-ATT
A classical attention baseline where the question representation attends over the answer hidden states.

In [ ]:
model_att = RNNATT(dummy_embed_matrix, hidden_dim=HIDDEN_DIM).to(device)
print(model_att)

### 3. RNN-POA
The complete Positional-Attention model using a Gaussian kernel to propagate positional influence.

In [ ]:
model_poa = RNNPOA(dummy_embed_matrix, hidden_dim=HIDDEN_DIM, sigma_scope=25, sigma_prime=0.1).to(device)
print(model_poa)

### Test Forward Pass

In [ ]:
B, L_q, L_a = 2, 10, 20
q_ids = torch.randint(1, VOCAB_SIZE, (B, L_q)).to(device)
a_ids = torch.randint(1, VOCAB_SIZE, (B, L_a)).to(device)
q_len = torch.tensor([L_q, L_q]).to(device)
a_len = torch.tensor([L_a, L_a]).to(device)
q_pos = [[2, 5], [1, 10]] # Example positions

sim_avg = model_avg(q_ids, a_ids, q_len, a_len, q_pos)
sim_att = model_att(q_ids, a_ids, q_len, a_len, q_pos)
sim_poa = model_poa(q_ids, a_ids, q_len, a_len, q_pos)

print('Output shapes (similarities):')
print('AVG:', sim_avg.shape)
print('ATT:', sim_att.shape)
print('POA:', sim_poa.shape)